# WhisperJAV Colab Edition v1.9.3 (Expert - Anime Optimized)

Put your videos in `Google Drive/WhisperJAV/`, then run Step 1 and Step 2.

In [ ]:
#@title Step 1: Expert Configuration { display-mode: "form" }

#@markdown ## Files and output
folder_name = "WhisperJAV" #@param {type:"string"}
subtitle_language = "Japanese" #@param ["Japanese", "English (Whisper direct)", "English (AI translate)"]

#@markdown ---
#@markdown ## 1. First pass (Optimized for Anime)
pass1_pipeline = "fidelity" #@param ["balanced", "fidelity", "fast", "faster", "transformers", "qwen"]
pass1_sensitivity = "aggressive" #@param ["conservative", "balanced", "aggressive"]

pass1_whisper_model = "large-v3" #@param ["automatic", "large-v2", "large-v3", "turbo"]
pass1_transformers_model = "kotoba-bilingual-v1.0" #@param ["kotoba-bilingual-v1.0", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"]
pass1_qwen_generator = "qwen3" #@param ["qwen3", "anime-whisper"]

pass1_builtin_vad = "automatic" #@param ["automatic", "3.1", "4.0", "6.2"]
pass1_speech_segmenter = "firered-vad" #@param ["automatic", "whisperseg", "firered-vad", "silero-v3.1", "silero-v4.0", "silero-v6.2", "nemo", "nemo-lite", "whisper-vad", "ten", "none"]

pass1_scene_detector = "semantic" #@param ["automatic", "semantic", "auditok", "silero", "none"]

pass1_speech_enhancer = "htdemucs" #@param ["none", "ffmpeg-dsp", "clearvoice", "zipenhancer", "bs-roformer", "htdemucs"]
pass1_ffmpeg_loudnorm = True #@param {type:"boolean"}
pass1_ffmpeg_normalize = False #@param {type:"boolean"}
pass1_ffmpeg_compress = False #@param {type:"boolean"}
pass1_ffmpeg_denoise = False #@param {type:"boolean"}
pass1_ffmpeg_highpass = False #@param {type:"boolean"}
pass1_ffmpeg_lowpass = False #@param {type:"boolean"}
pass1_ffmpeg_deess = False #@param {type:"boolean"}
pass1_ffmpeg_amplify = False #@param {type:"boolean"}

#@markdown ---
#@markdown ## Two passes
use_two_step = False #@param {type:"boolean"}
merge_method = "prefer first pass" #@param ["automatic", "keep all", "prefer first pass", "prefer second pass", "first pass on overlap", "second pass on overlap", "longest line"]

#@markdown ---
#@markdown ## 2. Second pass
pass2_pipeline = "qwen" #@param ["balanced", "fidelity", "fast", "faster", "transformers", "qwen"]
pass2_sensitivity = "aggressive" #@param ["conservative", "balanced", "aggressive"]
pass2_whisper_model = "automatic" #@param ["automatic", "large-v2", "large-v3", "turbo"]
pass2_transformers_model = "kotoba-bilingual-v1.0" #@param ["kotoba-bilingual-v1.0", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"]
pass2_qwen_generator = "qwen3" #@param ["qwen3", "anime-whisper"]
pass2_builtin_vad = "automatic" #@param ["automatic", "3.1", "4.0", "6.2"]
pass2_speech_segmenter = "automatic" #@param ["automatic", "whisperseg", "firered-vad", "silero-v3.1", "silero-v4.0", "silero-v6.2", "nemo", "nemo-lite", "whisper-vad", "ten", "none"]
pass2_scene_detector = "automatic" #@param ["automatic", "semantic", "auditok", "silero", "none"]
pass2_speech_enhancer = "none" #@param ["none", "ffmpeg-dsp", "clearvoice", "zipenhancer", "bs-roformer", "htdemucs"]
pass2_ffmpeg_loudnorm = True #@param {type:"boolean"}
pass2_ffmpeg_normalize = False #@param {type:"boolean"}
pass2_ffmpeg_compress = False #@param {type:"boolean"}
pass2_ffmpeg_denoise = False #@param {type:"boolean"}
pass2_ffmpeg_highpass = False #@param {type:"boolean"}
pass2_ffmpeg_lowpass = False #@param {type:"boolean"}
pass2_ffmpeg_deess = False #@param {type:"boolean"}
pass2_ffmpeg_amplify = False #@param {type:"boolean"}

#@markdown ---
#@markdown ## AI translation
translation_provider = "local" #@param ["local", "ollama", "custom", "deepseek", "openrouter", "gemini", "claude", "gpt", "glm", "groq"]
translation_target = "english" #@param ["english", "chinese", "korean", "indonesian", "thai", "spanish", "portuguese", "french", "italian"]
translation_tone = "standard" #@param ["standard", "contextual", "pornify"]
translation_model = "gemma-9b" #@param {type:"string"}
translation_endpoint = "" #@param {type:"string"}
api_key = "" #@param {type:"string"}

#@markdown ---
#@markdown ## Session
opening_credit = "" #@param {type:"string"}
closing_credit = "Subs by WhisperJAV" #@param {type:"string"}
auto_disconnect = True #@param {type:"boolean"}

WHISPERJAV_CMD = "whisperjav"
WHISPERJAV_TRANSLATE_CMD = "whisperjav-translate"
WHISPERJAV_RELEASE = "v1.9.3"
WHISPER_PIPELINES = {"balanced", "fidelity", "fast", "faster"}

merge_map = {
    "automatic": "smart_merge",
    "keep all": "full_merge",
    "prefer first pass": "pass1_primary",
    "prefer second pass": "pass2_primary",
    "first pass on overlap": "pass1_overlap",
    "second pass on overlap": "pass2_overlap",
    "longest line": "longest",
}
language_map = {
    "Japanese": "native",
    "English (Whisper direct)": "direct-to-english",
    "English (AI translate)": "llm",
}
transformers_model_map = {
    "kotoba-bilingual-v1.0": "kotoba-tech/kotoba-whisper-bilingual-v1.0",
    "kotoba-v2.0": "kotoba-tech/kotoba-whisper-v2.0",
    "kotoba-v2.1": "kotoba-tech/kotoba-whisper-v2.1",
    "kotoba-v2.2": "kotoba-tech/kotoba-whisper-v2.2",
}

warnings_list = []
errors_list = []

def pick(value):
    return None if value == "automatic" else value

def ffmpeg_filters(*chosen):
    names = ["loudnorm", "normalize", "compress", "denoise", "highpass", "lowpass", "deess", "amplify"]
    picked = [name for name, ticked in zip(names, chosen) if ticked]
    return ",".join(picked) if picked else "loudnorm"

def describe_pass(n, pipeline, sensitivity, whisper_model, transformers_model, qwen_generator, builtin_vad, segmenter, scene, enhancer):
    bits = [pipeline, sensitivity]
    if pipeline in WHISPER_PIPELINES and whisper_model != "automatic":
        bits.append(whisper_model)
    elif pipeline == "transformers":
        bits.append(transformers_model)
    elif pipeline == "qwen":
        bits.append(qwen_generator)
    if pipeline == "balanced":
        bits.append("built-in VAD " + ("default" if builtin_vad == "automatic" else builtin_vad))
    elif segmenter != "automatic":
        bits.append(segmenter)
    if scene != "automatic":
        bits.append("scene " + scene)
    if enhancer != "none":
        bits.append("clean-up " + enhancer)
    return " / ".join(bits)

WHISPERJAV_CONFIG = {
    "release": WHISPERJAV_RELEASE,
    "folder_name": folder_name,
    "subtitle_language": language_map[subtitle_language],
    "use_two_step": use_two_step,
    "merge_strategy": merge_map[merge_method],
    "pass1_pipeline": pass1_pipeline,
    "pass1_sensitivity": pass1_sensitivity,
    "pass1_whisper_model": pick(pass1_whisper_model),
    "pass1_transformers_model": transformers_model_map[pass1_transformers_model],
    "pass1_qwen_generator": pass1_qwen_generator,
    "pass1_builtin_vad": pick(pass1_builtin_vad),
    "pass1_speech_segmenter": pick(pass1_speech_segmenter),
    "pass1_scene_detector": pick(pass1_scene_detector),
    "pass1_speech_enhancer": None if pass1_speech_enhancer == "none" else pass1_speech_enhancer,
    "pass1_ffmpeg_filters": ffmpeg_filters(pass1_ffmpeg_loudnorm, pass1_ffmpeg_normalize, pass1_ffmpeg_compress, pass1_ffmpeg_denoise, pass1_ffmpeg_highpass, pass1_ffmpeg_lowpass, pass1_ffmpeg_deess, pass1_ffmpeg_amplify),
    "pass2_pipeline": pass2_pipeline,
    "pass2_sensitivity": pass2_sensitivity,
    "pass2_whisper_model": pick(pass2_whisper_model),
    "pass2_transformers_model": transformers_model_map[pass2_transformers_model],
    "pass2_qwen_generator": pass2_qwen_generator,
    "pass2_builtin_vad": pick(pass2_builtin_vad),
    "pass2_speech_segmenter": pick(pass2_speech_segmenter),
    "pass2_scene_detector": pick(pass2_scene_detector),
    "pass2_speech_enhancer": None if pass2_speech_enhancer == "none" else pass2_speech_enhancer,
    "pass2_ffmpeg_filters": ffmpeg_filters(pass2_ffmpeg_loudnorm, pass2_ffmpeg_normalize, pass2_ffmpeg_compress, pass2_ffmpeg_denoise, pass2_ffmpeg_highpass, pass2_ffmpeg_lowpass, pass2_ffmpeg_deess, pass2_ffmpeg_amplify),
    "translation_provider": translation_provider,
    "translation_target": translation_target,
    "translation_tone": translation_tone,
    "translation_model": translation_model.strip(),
    "translation_endpoint": translation_endpoint.strip(),
    "api_key": api_key,
    "opening_credit": opening_credit,
    "closing_credit": closing_credit,
    "auto_disconnect": auto_disconnect,
    "whisperjav_cmd": WHISPERJAV_CMD,
    "whisperjav_translate_cmd": WHISPERJAV_TRANSLATE_CMD,
    "_pass1_summary": describe_pass(1, pass1_pipeline, pass1_sensitivity, pass1_whisper_model, pass1_transformers_model, pass1_qwen_generator, pass1_builtin_vad, pass1_speech_segmenter, pass1_scene_detector, pass1_speech_enhancer),
    "_pass2_summary": describe_pass(2, pass2_pipeline, pass2_sensitivity, pass2_whisper_model, pass2_transformers_model, pass2_qwen_generator, pass2_builtin_vad, pass2_speech_segmenter, pass2_scene_detector, pass2_speech_enhancer),
    "_merge_method": merge_method,
}
WHISPERJAV_CONFIG_OK = not errors_list
from IPython.display import display, HTML
display(HTML('<div style="padding:10px;background:#f0f9ff;border-radius:4px;font-size:11px"><b>Ready</b> &nbsp; WhisperJAV Configured for Anime (Fidelity + FireRedVAD + Large-v3)</div>'))